In [0]:
%run ../utils/utils_feat_squad2_99_helpers

In [0]:
inicio = log_inicio("feat_squad2_01_connection_sql")

# ─────────────────────────────────────────────
# 1. TESTAR CONEXÃO
# ─────────────────────────────────────────────

try:
    df_test = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", "SELECT 1 AS conexao_ok")
        .load()
    )

    display(df_test)
    log.info("Conexão com SQL Server validada!")

except Exception as e:
    log.error(f"Erro na conexão: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 2. VERIFICAR SCHEMAS DISPONÍVEIS
# ─────────────────────────────────────────────

try:
    df_schemas = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT name AS schema_name
            FROM sys.schemas
            WHERE name NOT IN (
                'sys', 'INFORMATION_SCHEMA',
                'db_owner', 'db_accessadmin',
                'db_securityadmin', 'db_ddladmin',
                'db_backupoperator', 'db_datareader',
                'db_datawriter', 'db_denydatareader',
                'db_denydatawriter'
            )
        """)
        .load()
    )

    log.info("Schemas disponíveis:")
    display(df_schemas)

except Exception as e:
    log.error(f"Erro ao listar schemas: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 3. VERIFICAR TABELAS DO SQUAD 2
# ─────────────────────────────────────────────

try:
    df_tabelas = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT
                s.name AS schema_name,
                t.name AS table_name
            FROM sys.tables t
            JOIN sys.schemas s
                ON t.schema_id = s.schema_id
            WHERE s.name IN ('dbo', 'squad2')
            AND (
                t.name LIKE '%squad2%'
                OR t.name LIKE '%rastreamento%'
                OR t.name LIKE '%ecommerce%'
            )
        """)
        .load()
    )

    log.info("Tabelas relacionadas ao Squad 2 no banco:")
    display(df_tabelas)

except Exception as e:
    log.error(f"Erro ao listar tabelas: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 4. VERIFICAR STATUS DO SCHEMA SQUAD2
# ─────────────────────────────────────────────

try:
    df_schema = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT
                CASE
                    WHEN EXISTS (
                        SELECT 1
                        FROM sys.schemas
                        WHERE name = 'squad2'
                    )
                    THEN 'DISPONIVEL'
                    ELSE 'PENDENTE_CRIACAO'
                END AS squad2_status
        """)
        .load()
    )

    status = df_schema.collect()[0]["squad2_status"]

    if status == "DISPONIVEL":
        log.info("Schema squad2 disponível — pronto para uso.")
    else:
        log.warning(
            "Schema squad2 pendente — "
            "verifique com o responsável antes de gravar as tabelas."
        )

    display(df_schema)

except Exception as e:
    log.error(f"Erro ao verificar schema: {str(e)}")
    raise


log_fim("feat_squad2_01_connection_sql", inicio)

### Conexão com data lake no DB
Desenvolver um notebook de setup que monta (mount) ou conecta ao Azure Data Lake Gen2 utilizando as chaves de acesso (Service Principal ou Access Key), garantindo que você consiga listar os arquivos da camada Raw/Bronze.

In [0]:
pip install adlfs pandas python-dotenv

In [0]:
import os
import pandas as pd
import adlfs
from dotenv import load_dotenv

# Carregar .env
caminho_env = None
for tentativa in [".env", "../.env", "../../.env"]:
    if os.path.exists(tentativa):
        caminho_env = tentativa
        break

if caminho_env:
    load_dotenv(dotenv_path=caminho_env)
    print(f"✅ Arquivo .env carregado: {caminho_env}")
else:
    raise FileNotFoundError("⚠️ Arquivo .env não encontrado.")

In [0]:
# Credenciais ADLS
tenant_id = os.getenv("ADLS_TENANT_ID")
client_id = os.getenv("ADLS_CLIENT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account = "internshipdatalake"
container_name = "real-time-ecommerce-data"

if not tenant_id or not client_id or not client_secret:
    raise ValueError("⚠️ Credenciais do ADLS não encontradas no .env.")

os.environ["AZURE_TENANT_ID"] = tenant_id
os.environ["AZURE_CLIENT_ID"] = client_id
os.environ["AZURE_CLIENT_SECRET"] = client_secret

fs = adlfs.AzureBlobFileSystem(
    account_name=storage_account,
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

print("🔍 Procurando arquivo de rastreamento no Data Lake...")

todos_arquivos = fs.find(container_name)

caminho_real_no_azure = None
formato_detectado = None

for arquivo in todos_arquivos:
    nome = arquivo.lower()
    if "rastreamento" in nome:
        caminho_real_no_azure = arquivo
        formato_detectado = arquivo.split(".")[-1].lower()
        break

if not caminho_real_no_azure:
    raise FileNotFoundError("⚠️ Nenhum arquivo de rastreamento foi encontrado no Data Lake.")

caminho_final = f"abfs://{caminho_real_no_azure}"

print(f"✅ Arquivo localizado: {caminho_final}")
print(f"📦 Formato detectado: {formato_detectado.upper()}")

In [0]:
# Leitura dinâmica com Pandas
credenciais_pandas = {
    "account_name": storage_account,
    "client_id": client_id,
    "client_secret": client_secret,
    "tenant_id": tenant_id
}

print(f"📥 Lendo arquivo no formato {formato_detectado.upper()}...")

if formato_detectado == "parquet":
    df_pandas = pd.read_parquet(caminho_final, storage_options=credenciais_pandas)

elif formato_detectado == "csv":
    df_pandas = pd.read_csv(caminho_final, storage_options=credenciais_pandas)

elif formato_detectado == "json":
    df_pandas = pd.read_json(caminho_final, storage_options=credenciais_pandas)

else:
    raise TypeError(f"⚠️ Formato não suportado: {formato_detectado}")

df_rastreamento = spark.createDataFrame(df_pandas)

print("✅ Dados carregados no Spark!")
display(df_rastreamento)
df_rastreamento.printSchema()

In [0]:
# Configuração SQL Server
jdbc_host = os.getenv("SQL_HOST")
jdbc_db = os.getenv("SQL_DATABASE")
jdbc_user = os.getenv("SQL_USERNAME")
jdbc_pass = os.getenv("SQL_PASSWORD")

nome_tabela_destino = "squad2.ecommerce_rastreamento"

print(f"📤 Gravando dados na tabela: {nome_tabela_destino}")

(
    df_rastreamento.write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_db)
    .option("user", jdbc_user)
    .option("password", jdbc_pass)
    .option("dbtable", nome_tabela_destino)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .mode("overwrite")
    .save()
)

print("🚀 Tabela squad2.ecommerce_rastreamento criada com sucesso no SQL Server!")